# KHIS Data Quality Walkthrough

This notebook shows how to assess county-style DHIS2/KHIS data quality from end to end. We connect to the public DHIS2 demo server, pull malaria-related data for a few organisation units, clean it, and then run the quality scorecard that county health teams can use during review meetings.

In [ ]:
import khis
import pandas as pd
from IPython.display import display

conn = khis.connect()
status, message = conn.ping()
print(message)
print("Using demo server:" , conn.using_demo_server)

In [ ]:
indicator_matches = khis.get_demo_indicators(conn, search_term="malaria")
display(indicator_matches.head())

indicator_id = indicator_matches.iloc[0]["id"]
indicator_name = indicator_matches.iloc[0]["name"]
selected_org_units = khis.get_demo_org_units(conn, limit=3)
display(selected_org_units)

df_raw = khis.build_demo_indicator_frame(
    conn,
    indicator_id=indicator_id,
    indicator_name=indicator_name,
    org_units=selected_org_units,
    periods=khis.DEMO_ANALYTICS_PERIODS,
)
df_clean = khis.clean(df_raw)
df_clean.head()

## What We Are Checking

Routine county data can fail in different ways. Some counties may not report every month, some values may be unusually high or low, some reports may arrive late, and some long runs of zeros may actually be missing entries. The next cells break those checks apart before combining them into one scorecard.

In [ ]:
completeness = khis.completeness_score(df_clean, expected_periods=12)
display(
    completeness.sort_values(["county", "indicator"]).style.background_gradient(
        subset=["completeness_pct"], cmap="RdYlGn"
    )
)

In [ ]:
outliers = khis.outlier_report(df_clean)
display(outliers[outliers["outlier_flag"]].head(10))

In [ ]:
scorecard, summary = khis.quality_report(df_clean)
display(
    scorecard.style.background_gradient(subset=["completeness_score"], cmap="RdYlGn")
)
print(summary)

heatmap = khis.plot_quality_heatmap(completeness)
heatmap

## Interpretation Guide

- **Completeness** tells you whether a county or organisation unit reported enough periods to support trend analysis.
- **Outliers** highlight values that should be checked against source registers or facility reports before action is taken.
- **Timeliness** matters because a report that arrives very late may be complete but still not useful for rapid county response.
- **Suspicious zeros** matter because repeated zeros can hide non-reporting, especially when the same series sometimes records real non-zero values.

## What To Do With This Information

A county health officer can use this scorecard to decide where to follow up first. Counties or sub-counties with poor completeness or suspicious zero patterns need data review before the figures are used in planning. Outliers should be discussed with facilities, while late reporters may need supportive supervision, reminders, or workflow changes so the next reporting cycle is more reliable.